# sphinx_copybutton

sphinx-copybutton 0.5.2 adds a copy button to every code block of a Sphinx site. These cells build small projects in a
temporary folder and check what the extension writes: the buttons' script and style, its configuration values carried
into the page's script, the exclude and selector settings, and a strict build.

In [ ]:
# Example 1: the extension copies its script, style and clipboard helper and adds them to the page.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_copybutton']\n")
(src / "index.rst").write_text("Title\n=====\n\n.. code-block:: python\n\n   print('hello')\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
static = src / "_out" / "_static"
html = (src / "_out" / "index.html").read_text()
print("rc", rc, sorted(p.name for p in static.glob("c*")))
assert rc == 0
for name in ("copybutton.js", "copybutton.css", "clipboard.min.js", "copybutton_funcs.js"):
    assert (static / name).is_file(), name
assert "clipboard.min.js" in html and "copybutton.js" in html and "copybutton.css" in html

In [ ]:
# Example 2: the prompt-removal settings reach the page's script.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_copybutton']\ncopybutton_prompt_text = '>>> '\ncopybutton_line_continuation_character = '\\\\'\n")
(src / "index.rst").write_text("Title\n=====\n\n.. code-block:: console\n\n   >>> ls\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
js = (src / "_out" / "_static" / "copybutton.js").read_text()
print(js[js.find("copybuttonPromptText") if "copybuttonPromptText" in js else 0:][:120])
assert rc == 0
assert "'>>> '" in js or '">>> "' in js, "the prompt text did not reach copybutton.js"

In [ ]:
# Example 3: copybutton_exclude: the setting is carried into the script that skips these elements.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_copybutton']\ncopybutton_exclude = '.linenos, .gp, .noselect'\n")
(src / "index.rst").write_text("Title\n=====\n\n::\n\n   text\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
js = (src / "_out" / "_static" / "copybutton.js").read_text()
print("exclude present:", ".noselect" in js)
assert rc == 0
assert ".linenos, .gp, .noselect" in js

In [ ]:
# Example 4: copybutton_selector chooses which blocks get a button; the default selector is replaced.
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_copybutton']\ncopybutton_selector = 'div.only-these pre'\n")
(src / "index.rst").write_text("Title\n=====\n\n.. code-block:: python\n\n   x = 1\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
js = (src / "_out" / "_static" / "copybutton.js").read_text()
assert rc == 0
assert "div.only-these pre" in js
assert "div.highlight pre" not in js, "the default selector was not replaced"
print("selector ok")

In [ ]:
# Example 5: the extension declares itself parallel-safe and survives a strict two-worker build.
import pathlib, tempfile
from unittest import mock
from sphinx.cmd.build import build_main
import sphinx_copybutton

info = sphinx_copybutton.setup(mock.MagicMock())  # the entry point Sphinx calls for an extension
assert info["parallel_read_safe"] is True and info["parallel_write_safe"] is True, info

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("extensions = ['sphinx_copybutton']\n")
pages = "\n".join(f"   page{i}" for i in range(8))
(src / "index.rst").write_text(f"Title\n=====\n\n.. toctree::\n\n{pages}\n")
for i in range(8):
    (src / f"page{i}.rst").write_text(f"Page {i}\n=======\n\n.. code-block:: python\n\n   print({i})\n")
rc = build_main(["-q", "-W", "-j", "2", "-b", "html", str(src), str(src / "_out")])
print("rc", rc, sphinx_copybutton.__version__)
assert rc == 0
assert sphinx_copybutton.__version__ == "0.5.2"
assert (src / "_out" / "page7.html").is_file()